# Introduction

This is the code submission for group 7, called "Onsenfoutrouveunnom" on kaggle, by Bernet Théo (SCIPER: 392857) and Labaeye Justin (SCIPER: 400868).


### Goal of the model:

Considering a soil around a nuclear waste canister, we'll use a data-driven method to predict the temperature of the rocks at specific locations for given times and powers.

### Data at disposal:

Three parquet files, including:
- sensors: set including all coordinates of test and train sensors
- train: training set, with sensor names, times, powers, and temperature as columns
- test: test set, of the same structure as "train", but without temperature

### Our Model

We implemented a K-nearest neighbors model to predict the temperatures of the test sensors.
A linear regression model was also intended in the first place, to take over the KNN, but the project was finally aborted.

The main idea is the following: for each sensor to predict,
- find its neighbors,
- compute for each a "score", which is an inverse function of the distance,
- calculate the mean value of each neighbors' temperature weighted by their score, for each (time / power) couple.

In addition to that, it is necessary to preprocess the train data before using it.
In fact, in the set lie a significant amount of misleading sensors, and outliers values. It is crucial to identify them, and delete and/or replace such values.

## Imports

In [ ]:
"""Inserts"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time as tm
from typing import Any, Callable
import warnings
warnings.filterwarnings("ignore")

# KNN Model

### 1. Distance Metrics

In [ ]:
def man_dist(sample_coo: pd.DataFrame, train_set_coo: pd.DataFrame) -> pd.DataFrame:

    """Computes the Manhattan distance between a sample and all train sensors, and add
    their values in a new column of train_set_coo.
    Args:
        sample_coo: Dataset of the sample's coordinates
        train_set_coo: Dataset of the train sensors' coordinates
    Returns:
        train_set_coo_with_dist: train_set_coo with a column "distance", filled.
    """

    coor_train = np.array([train_set_coo["coor_x"], train_set_coo["coor_y"]]).T
    coor_sample = np.array([sample_coo["coor_x"], sample_coo["coor_y"]])

    train_set_coo["distance"] = np.abs(coor_train - coor_sample).sum(axis = 1)

    return train_set_coo
   


def eucli_dist(sample_coo: pd.DataFrame, train_set_coo: pd.DataFrame) -> pd.DataFrame:

    """Computes the Euclidian distance between a sample and all train sensors, and add
    their values in a new column of train_set_coo.
    Args:
        sample_coo: Dataset of the sample's coordinates
        train_set_coo: Dataset of the train sensors' coordinates
    Returns:
        train_set_coo_with_dist: train_set_coo with a column "distance", filled.
    """
    
    coor_train = np.array([train_set_coo["coor_x"], train_set_coo["coor_y"]]).T
    coor_sample = np.array([sample_coo["coor_x"], sample_coo["coor_y"]])

    train_set_coo["distance"] = np.sqrt(((coor_train - coor_sample) ** 2).sum(axis = 1))

    return train_set_coo


### 2. Distance Score

In [ ]:
def distance_score(neighbors:pd.DataFrame,
                   score_parameter: float = 0.1) -> pd.DataFrame:
    
    """Replace the "distance" column by a "score" one to each given neighbor sensor,
    depending on their distance to the sample.
    The closer the sample, the higher the score. We chose our score equation such that
    we avoid giving "blind confidence" to "very close" sensors.
    Args:
        neighbors: neighbor sensors, with "distance" column.
        score_parameter: Hyper-parameter, set to 1 by default.
    Returns:
        neighbors: Updated with the "score" column.
    """

    scores = 1 / (score_parameter * neighbors["distance"] + 1)

    # We replace the distance column (which we don't need anymore) by the "score" one.
    neighbors["score"] = scores
    neighbors.drop(labels = ["distance"], axis = 1, inplace = True)
    
    return neighbors

### 3. K-Nearest Neighbors

In [ ]:
def find_nearest_neighbors(
    sample_coo: pd.DataFrame, 
    train_set_coo: pd.DataFrame, 
    distance_fn: Callable = eucli_dist, 
    k: int = 7,
    score_parameter: int = 1) -> pd.DataFrame:
    
    """Finds the names and distance-scores of the k-Nearest Neighbors of a given sample.
    Args:
        sample_coo: Coordinates of the sample.
        train_set_coo: Dataset of the the sensors' coordinates.
        distance_fn: Distance function.
        k: Number of nearest neighbors considered, set to 7 by default.
        score_parameter: Hyper-parameter of distance score, set to 1 by default.
    Returns:
        neighbors: dataframe of the k-nearest neighbors of the sample, with their distance's scores
         stored in a new "score" column.
    """

    train_set_with_dist = distance_fn(sample_coo, train_set_coo)
    
    neighbors = distance_score(neighbors= train_set_with_dist.sort_values(by=["distance"]).head(k),
                               score_parameter= score_parameter)
    
    return neighbors

### 4. Prediction Functions

In [ ]:
def prediction_single(
    times_to_pred : np.ndarray,
    neighbors_values: pd.DataFrame,
    neighbors_scores: pd.DataFrame) -> list:

    """Gives the final prediction of a given sample's temperatures, for all powers and times.
    Calculates the mean value of the temperatures of the neighbors, weighted by their distance scores.
    Args:
        times_to_pred: All times at which we'll make a prediction.
        neighbors_values: The dataframe of values of the k-nearest neighbors of the sample.
        neighbors_scores: The dataframe of scores and coordinates of the neighbors.
    Returns:
        pred: The values of the predicted temperatures of the sample, for all powers and times.
    """

    pred = []
    total_scores = np.sum(neighbors_scores["score"])
    
    for pow_tier in range(3):
        rows = neighbors_values[neighbors_values["tier"] == pow_tier]

        for time in times_to_pred:
            mask = (rows.time == time)
            temperatures = rows.loc[mask, "temperature"]
            pred.append(np.sum(temperatures * neighbors_scores["score"]) / total_scores)

    return pred



def prediction(
    validation_set_coo: pd.DataFrame,
    validation_set_values:pd.DataFrame,
    train_set_coo: pd.DataFrame,
    train_set_values: pd.DataFrame,
    distance_fn: Callable = eucli_dist, 
    k: int = 7,
    score_parameter: int = 1) -> pd.DataFrame:

    """Gives the final predictions of all validation-set temperatures, while calling prediction_single.
    Args:
        validation_set_coo: Dataframe with names and coordinates of some sensors whom we "hide" the
         temperature values.
        validation_set_values: Dataframe with names, times and powers of the validation set at which we want the
         model to evaluate the temperature (times with "not nan" temperatures)
        train_set_coo: Dataset of the "train" sensors' coordinates.
        train_set_values: Dataset of the "train" sensors' values of time, power and temperature.
        distance_fn: Distance function (among Manhattan or Euclidian)
        k: Number of nearest neighbors taken, set to 7 by default
        score_parameter: Hyper-parameter of distance score, set to 1 by default.

    Returns:
        pred: validation_set_values with a column "prediction", filled with the outputs of prediction_single.
    """

    # Empty array in which we'll store our predictions
    validation_set_values["prediction"] =  np.empty(validation_set_values.shape[0])

    # All times at which we'll make a prediction
    times_to_pred = validation_set_values["time"].unique()

    for sensor_name, row in validation_set_coo.iterrows():

        start = tm.time() # We'll evaluate the time our model takes to predict the temperature of each sensor.
        
        neighbors = find_nearest_neighbors(sample_coo = row,
                                           train_set_coo = train_set_coo,
                                           distance_fn = distance_fn,
                                           k = k, 
                                           score_parameter=score_parameter)
        
        validation_set_values.loc[sensor_name,"prediction"] = prediction_single(
                                                   times_to_pred=times_to_pred ,
                                                   neighbors_values= train_set_values.loc[neighbors.index],
                                                   neighbors_scores= neighbors)
        
        end = tm.time()
        print(f"Prediction de {sensor_name} terminée. Temps: {end - start}")
    
    return validation_set_values

# Running section

First, let's read the data. We set every indices to the "sensor" column for convenience.

In [ ]:
training_data=pd.read_parquet("data_parquet_2026/train.parquet").set_index(keys= "sensor")
sensors = pd.read_parquet("data_parquet_2026/sensors.parquet").drop(columns=["coor_z"]).drop_duplicates().set_index(keys= "sensor")
test_data = pd.read_parquet("data_parquet_2026/test.parquet").set_index(keys= "sensor")

Now, let's create a "tier" column, to differentiate each "temperature behavior", which depends on their power. Separating the data into 3 classes will help us for treatment.

In [ ]:
# We create a new column to the training_data set, to identify the "power-function" corresponding to each row.
training_data["tier"] = np.empty(training_data.shape[0],dtype=int)

# The number of rows of a given sensor, for each power
n = int(training_data.shape[0] / training_data.index.unique().shape[0] / 3 )

for senso in training_data.index.unique():
    stack = [0]*n + [1]*n + [2]*n    
    training_data.loc[senso, "tier"] = stack

Now, let's plot our data to see how it looks like.

In [ ]:
training_data.plot.scatter(x="time",y="temperature",alpha=0.5)

## Preprocessing

Idea for the preprocessing: for each sensor, compute the median and standart error on a rolling window, and replace out-of-boundaries values by the median.

In [ ]:
def window_cleaning(rows: pd.DataFrame, margin: float = 1.5, window : int = 50) -> pd.DataFrame:

    """Cleans a set of rows by computing the median, and cutting the values under and above a
    certain distance of it (corresponding to margin-times the standart error) and replacing
    these values by the median.
    Args:
        rows: the rows to clean.
        margin: coefficient of the standart error to create an interval of "okay" values.
        window: number of rows considered at each iteration to compute the median and standart error.
    Returns:
        rows: same rows with replaced outliers.
    """
    
    n = rows.shape[0]

    # We need numbers as index to differentiate each row. (currently ["sensor","tier"])
    rows.reset_index(inplace=True)

    for t in range(window, n + window , window):
        
        # We "come back" a little at the end to make a whole window on the last rows.
        if t >= n:
            t = n
        
        mask = ((t-window)*864000 <= rows["time"])*( rows["time"] < t*864000)
        window_rows = rows.loc[mask]
        
        med = np.median(window_rows.dropna()["temperature"])
        std_error = np.std(window_rows.dropna()["temperature"])
        low_bound = med - margin*std_error
        up_bound = med + margin*std_error
        
        outlier_index = window_rows.index[
        (window_rows["temperature"] < low_bound) | 
        (window_rows["temperature"] > up_bound) | 
        (window_rows["temperature"].isna())]
        
        rows.loc[outlier_index, "temperature"] = med
        
    return rows


def preprocessing(sensors_set: pd.DataFrame, window: int = 50, margin: float = 1.5) -> pd.DataFrame:

    """The global function to preprocess the Data. It cleans each sensor one by one by looking
    at 'windows' of time values on their data, computing the median on such intervals, and
    replacing too-far values of the median by itself.
    Args:
        sensors_set: dataset of all sensors to preprocess, with values of time,
         power and temperature.
        window: amount of rows to take to compute the median.
        margin: size of the safe zone, where the values are kept around the median.
    Returns:
        sensors_set: with cleaner data
    """
    
    dropped = []

    # We pick every specific sensor AND power tier of the dataset.
    for sample in sensors_set.index.unique(): # sample == ["sensor", "tier"]

        if sample[0] in dropped:
            continue

        all_rows = sensors_set.loc[sample] # All the temperatures of a given sample and power tier

        # We drop the drifting sensors, whose last values are out of a peculiar range (here [0; 25])
        if np.mean(all_rows.dropna()["temperature"][:-5:-1]) < 0 or np.mean(all_rows["temperature"][:-5:-1]) > 25:
            sensors_set.drop(sample[0],inplace=True)
            print(f"Sample {sample[0]} dropped")
            dropped.append (sample[0])
    
        else:
            cleaned = window_cleaning(rows= all_rows, margin= margin, window= window).set_index(["sensor","tier"])
            sensors_set.loc[sample] = cleaned
    
    return sensors_set

Now, let's run it.

In [ ]:
training_data = preprocessing(training_data.set_index("tier", append=True))


"""Plot the new training set"""
training_data.plot.scatter(x="time",y="temperature",alpha=0.5)

"""for each power"""
training_data.reset_index(inplace=True) # "tier" is currently in the index
p1=training_data[training_data["tier"]==0]
p2=training_data[training_data["tier"]==1]
p3=training_data[training_data["tier"]==2]
p1.plot.scatter(x="time",y="temperature",alpha=0.5)
p2.plot.scatter(x="time",y="temperature",alpha=0.5)
p3.plot.scatter(x="time",y="temperature",alpha=0.5)

## Training

Now, we want to find an optimal value for k. First, let's define a loss to evaluate it.

In [ ]:
def loss(ground_truths: pd.DataFrame, prediction_set: pd.DataFrame) -> float:
    
    """ Computes the mean error of the model to evaluate it.
    Args:
        ground_truths: Dataframe of the name of the sensors with their time, and real temperatures.
        prediction_set: Our "Prediction" dataframe, of the same structure as ground_truths, but with
         a column "prediction" instead of "temperature".
    Returns:
        float: The loss
    """
    
    real_temp = ground_truths["temperature"]
    predictions = prediction_set["prediction"]

    return np.mean(np.abs(real_temp - predictions))

And let's prepare our data to put it in our prediction() function.

In [ ]:
"""Dropping the test sensors, as we will train the model only with the training data"""
training_data.set_index("sensor",inplace=True)
test_sensors = test_data.index.unique()
train_sensors = sensors.loc[training_data.index.unique()]


"""Partition of the training_data into our "training" and "validation" model's sets""" 
validation_sensors_coo = train_sensors.sample(frac = 1/5)
train_sensors_coo = train_sensors.drop(labels=validation_sensors_coo.index)

validation_samples_with_temp = training_data.loc[validation_sensors_coo.index]
train_samples_with_temp = training_data.drop(validation_sensors_coo.index)

Now let's run our model!

In [ ]:
losses=[]
k_values=[]

for k in range(5, 10):

    validation_samples_with_pred = prediction(
        validation_set_coo = validation_sensors_coo,
        validation_set_values = validation_samples_with_temp,
        train_set_coo = train_sensors_coo,
        train_set_values = train_samples_with_temp,
        distance_fn = eucli_dist, 
        k = k,
        score_parameter=1)
    
    get_loss = loss(
        ground_truths = validation_samples_with_temp.dropna(),
        prediction_set= validation_samples_with_pred.dropna())
    k_values.append(k)
    losses.append(get_loss)
    print(f"Value of k: {k} \n Loss: {get_loss} \n --------- \n")
    
plt.scatter(k_values,losses)

A value of k = 5 seems to work good. We'll use this value for the test.

NOTE: If you run the code, you may found another value to be the best for k. This is due to the random sampling of the validation set: the optimal value of k depends on the sensors chosen to evaluate it. In order to have a "general" optimal value of k, we would need a n-folding function, to repeat the process on n different validation sets, and compare the results of each sampling. However, as it would be n-times longer (n predictions for each value of k), we avoid it for you here, and ask you to believe that 5 is indeed an optimal value for k.

## Test

In [ ]:
sub = prediction(
        validation_set_coo = sensors.loc[test_sensors],
        validation_set_values = test_data,
        train_set_coo = sensors.loc[train_sensors.index],
        train_set_values = training_data,
        distance_fn = eucli_dist, 
        k = 5,
        score_parameter=7)

In [ ]:
submission = pd.DataFrame({
   "Id": np.arange(len(test_data), dtype=int),
   "temperature": sub["prediction"],
})
assert list(submission.columns) == ["Id", "temperature"]
assert len(submission) == len(test_data)
assert (submission["Id"].to_numpy() == np.arange(len(test_data))).all()
assert np.isfinite(submission["temperature"]).all()
assert submission.isna().sum().sum() == 0
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

# Final notes

We didn't run the plotting parts in the submission, as we were scared the file would be too large for submitting. You'll have to run everything to see them.

Also, we only optimize k here. We didn't keep all our previous versions of the model (preprocessing with only fillna(), preprocessing without sensors drift deletion, score_parameter optimization, etc.).

The "proofs" of our improvings are heared for us as the different submission scores we obtained. The comparisions between each version are done on the poster.

Thank you for getting there!
Hope everything was clear enough, and worked well for you.
Best,
Théo and Justin